# Проектирование A/B-теста: нейросетевой ранкер vs CatBoost


## 1. Контекст и цель

Онлайн-кинотеатр. Приложение на смартфоны. В разделе «Что смотреть» реализована бесконечная лента рекомендаций. Сейчас ранжирование фильмов и сериалов выполняется с помощью модели CatBoost. Команда предполагает, что нейросетевой ранкер может лучше улавливать сложные зависимости в поведении пользователей и повысить релевантность рекомендаций.

**Цель A/B-теста** — проверить, улучшает ли нейросетевой ранкер ключевую продуктовую метрику (CTR рекомендаций) по сравнению с текущим CatBoost, не ухудшая guardrail-метрики (время загрузки, ошибки, отток пользователей).

Результат теста поможет принять решение: внедрять ли нейросетевой ранкер в продакшен или оставить CatBoost.

> **Примечание:** baseline, MDE и трафик ниже являются сценарными допущениями для портфолио; исторические данные реального онлайн-кинотеатра в проекте отсутствуют.


## 2. Гипотеза

Если заменить текущий ранкер на CatBoost на нейросетевой ранкер в разделе «Что смотреть», то CTR рекомендаций вырастет, потому что нейросеть лучше улавливает нелинейные зависимости между историей просмотров, контекстом и признаками контента, чем градиентный бустинг на табличных признаках.

**Статистическая постановка:**
- **H0:** CTR рекомендаций в группе B не отличается от группы A.
- **H1:** CTR рекомендаций в группах A и B отличается.

**Бизнес-гипотеза:** для успешного результата CTR в группе B должен вырасти минимум на **+5% относительно**. Меньший прирост не оправдывает стоимость внедрения и поддержки новой модели.

**Ограничение гипотезы:** нейросетевой ранкер не должен ухудшать guardrail-метрики (время загрузки ленты, доля ошибок, отток).


## 3. Метрики

### 3.1 Primary

**CTR рекомендаций** — доля пользователей, которые кликнули хотя бы на один рекомендованный контент, от числа пользователей, увидевших ленту.

Формула: `CTR = число уникальных пользователей с кликом / число уникальных пользователей с показом ленты`

Почему именно она: это основная продуктовая цель раздела «Что смотреть» — заинтересовать пользователя и привести его к просмотру. Метрика чувствительна к качеству ранжирования и напрямую связана с гипотезой.

### 3.2 Secondary

- **Средняя глубина скролла** — сколько карточек пользователь просматривает за сессию. Показывает вовлечённость в ленту.
- **Число просмотров на пользователя** — помогает проверить, приводит ли изменение CTR к большему числу фактических просмотров.
- **Время до первого клика** — косвенно показывает релевантность верхних позиций ленты.

Secondary-метрики не используются для принятия решения, но помогают интерпретировать результат.

### 3.3 Guardrail

- **Время загрузки ленты (p95)** — нейросетевой ранкер может быть тяжелее CatBoost.
- **Доля ошибок при формировании рекомендаций** — техническая стабильность.
- **Отток пользователей (7-day retention)** — не должен падать.
- **Число жалоб на рекомендации** — контроль качества контента.

Для guardrail-метрик до запуска фиксируется допустимое ухудшение. Статистическая значимость сама по себе не является достаточным основанием: при интерпретации учитывается и практическая величина эффекта.

### 3.4 Единица анализа

Единица анализа — **пользователь** (`user_id`), а не сессия или показ. Это соответствует рандомизации и не позволяет одному активному пользователю многократно увеличивать вес своего поведения.


## 4. Дизайн эксперимента

**A (контроль):** текущий ранкер на CatBoost.  
**B (тест):** нейросетевой ранкер.

Всё остальное в ленте «Что смотреть» остаётся неизменным: верстка, число карточек, механика скролла, алгоритм формирования пула кандидатов. Меняется только модель ранжирования.

**Единица рандомизации:** пользователь (`user_id`). Один пользователь попадает только в одну группу на всё время теста.

**Почему по `user_id`:** рекомендации — это пользовательский опыт, а поведение одного пользователя коррелирует между сессиями. Рандомизация по сессии или показу создаёт зависимые наблюдения и может завышать статистическую значимость.

**Разделение трафика:** 50% / 50%.

**Стабильность назначения:** вариант фиксируется за пользователем на весь период теста (через хэш от `user_id`), чтобы исключить эффект переключения и загрязнение групп.

**Целевая аудитория:** все пользователи, которые заходят в раздел «Что смотреть».

**Исключения:**
- боты и автоматические запросы;
- сотрудники компании.

**Длительность:** минимум — 2 полных недели, чтобы захватить недельные циклы активности.

**Запуск:** одновременно для всех пользователей, без постепенного раската.

**Контроль качества после запуска:** проверка SRM (Sample Ratio Mismatch) — фактическое соотношение групп не должно значимо отличаться от 50/50. При обнаружении SRM тест останавливается и разбирается причина.


## 5. Логирование

Для анализа необходимо сохранять:

- `user_id`;
- вариант эксперимента (`A` / `B`);
- показ ленты;
- клик по карточке;
- начало просмотра;
- время загрузки ленты и ошибки.

Вариант эксперимента фиксируется на уровне пользователя и записывается во все связанные события. Это позволяет связать пользовательское поведение с назначенным вариантом и проверить корректность данных после завершения теста.


## 6. Расчёт размера выборки

### 6.1 Вводные

Для расчёта нужны четыре параметра:

- **Baseline CTR (группа A):** 20% — доля пользователей, кликнувших хотя бы на одну карточку в ленте «Что смотреть».
- **Минимальный детектируемый эффект (MDE):** +5% относительно, то есть рост CTR с 20% до 21%. Меньший прирост не оправдывает стоимость внедрения и поддержки нейросетевого ранкера.
- **Уровень значимости (α):** 0.05, двусторонний тест. Допускаем 5% ложноположительных срабатываний.
- **Мощность (1 − β):** 0.80. В 80% случаев тест должен обнаружить эффект, если он действительно есть.

Дополнительно известен **дневной трафик раздела** — 200 000 уникальных пользователей в день. Он нужен, чтобы перевести размер выборки в длительность теста.

### 6.2 Метод

Используется стандартная формула для сравнения двух пропорций (двухвыборочный z-тест):

n = ((z_{1−α/2} + z_{1−β})² · (p₁(1−p₁) + p₂(1−p₂))) / (p₂ − p₁)²

где p₁ — baseline CTR, p₂ — ожидаемый CTR в тесте, z — квантили нормального распределения. Формула даёт размер выборки **на одну группу**; общий размер — удвоенное значение.


In [1]:
from math import ceil
from scipy.stats import norm

p1 = 0.20          # baseline CTR
mde = 0.05         # относительный MDE
p2 = p1 * (1 + mde)

alpha = 0.05
power = 0.80

z_alpha = norm.ppf(1 - alpha / 2)
z_beta = norm.ppf(power)

n = ((z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2))) / (p2 - p1) ** 2
n = ceil(n)

print(f"Размер выборки на группу: {n:,} пользователей")
print(f"Общий размер: {2 * n:,} пользователей")

daily_traffic = 200_000
days = ceil((2 * n) / daily_traffic)
print(f"Минимальная длительность по выборке: {days} день")


Размер выборки на группу: 25,580 пользователей
Общий размер: 51,160 пользователей
Минимальная длительность по выборке: 1 день


### 6.3 Результат расчёта

При указанных вводных:

- **Размер выборки на группу:** 25 580 пользователей.
- **Общий размер:** 51 160 пользователей.
- **Минимальная длительность при трафике 200 000 польз./день:** 1 день.

То есть формально расчётная выборка набирается за один день. Это не означает, что тест следует завершать через один день.

### 6.4 Итоговая длительность

Несмотря на расчётный размер выборки, тест запускается на **2 полных недели**. Причины:

- **Недельные циклы.** Поведение пользователей в будни и выходные различается. Тест должен включить несколько полных недель, а не один календарный отрезок.
- **Проверка устойчивости эффекта.** Первые дни могут отличаться от последующих из-за изменения пользовательского поведения после запуска новой модели; две недели дают возможность наблюдать эффект в разных периодах.
- **Стабильность guardrail-метрик.** Время загрузки, долю ошибок и 7-day retention нужно наблюдать на реальной нагрузке в течение нескольких дней, включая пиковые часы.
- **Запас на SRM и технические сбои.** Если часть данных придётся исключить из-за проблем с логированием или распределением трафика, запас по длительности компенсирует потери.

**Дозревание retention:** после остановки теста данные по `7-day retention` собираются ещё 7 дней; финальный анализ проводится после завершения этого окна.

**Финальное правило:** минимум 2 полных недели, даже если расчётная выборка набирается быстрее.


## 7. План статистического анализа

Анализ проводится после завершения теста, на полных данных. Порядок шагов важен: сначала проверяем валидность данных, потом основную метрику, потом всё остальное.

### 7.1 Проверка качества данных (SRM)

Первым делом проверяется **Sample Ratio Mismatch** — соответствие фактического распределения пользователей по группам ожидаемому 50/50.

- Метод: хи-квадрат (χ²) на наблюдаемых частотах.
- Критерий: p-value < 0.001 считается признаком SRM (строгий порог, чтобы не срабатывать на небольших случайных колебаниях).
- Если SRM обнаружен — тест **не анализируется**, сначала ищется причина: баг в рандомизации, потеря данных, фильтрация ботов, задевшая одну группу сильнее другой.

Дополнительно проверяется, что исключения (боты, сотрудники) применены симметрично к обеим группам.

### 7.2 Primary-метрика

**CTR рекомендаций** сравнивается между группами A и B.

- **Метод:** двухвыборочный z-тест для пропорций (двусторонний).
- **Уровень значимости:** α = 0.05.
- **Что считаем:** абсолютную разницу CTR, относительный uplift (%), 95% доверительный интервал для разницы.
- **Критерий значимости:** p-value < 0.05 и 95% доверительный интервал для разницы не пересекает ноль.

Результат интерпретируется вместе с размером эффекта: статистически значимое изменение само по себе не означает бизнес-успех; для положительного решения нужен эффект не ниже заранее заданного MDE.

### 7.3 Secondary-метрики

- **Средняя глубина скролла** — t-тест Уэлча (не предполагаем равенство дисперсий).
- **Число просмотров на пользователя** — t-тест Уэлча или бутстрап, если распределение сильно скошено.
- **Время до первого клика** — бутстрап, так как распределение времени обычно не нормальное.

Эти метрики не влияют на решение напрямую, но помогают объяснить результат: например, CTR вырос, а число просмотров не выросло — значит, дополнительные клики не привели к дополнительному потреблению контента.

### 7.4 Guardrail-метрики

Проверяются на ухудшение, а не на улучшение.

- **Время загрузки ленты (p95)** — сравнение квантилей через бутстрап.
- **Доля ошибок** — z-тест для пропорций.
- **Отток (7-day retention)** — z-тест для пропорций.
- **Число жалоб** — сравнение частот.

Критерий: если guardrail-метрика значимо ухудшилась (p < 0.05) и ухудшение превышает заранее зафиксированный практический порог, тест считается провальным независимо от роста CTR.

### 7.5 Множественные сравнения

Основная метрика — одна, по ней принимается решение, поэтому поправка на множественность (Бонферрони и т.п.) для primary-метрики не применяется. Secondary- и guardrail-метрики используются вспомогательно; их p-value интерпретируются как описательные, а не как самостоятельные основания для продуктового вывода.

### 7.6 Сегментный анализ (опционально)

Если есть ресурс и результат по основной метрике неоднозначный, дополнительно смотрим срезы:

- новые vs вернувшиеся пользователи;
- активность пользователя (число сессий за период).

Сегментный анализ делается **только как исследовательский**, без права принимать по нему решение.


## 8. Критерии принятия решения

| Сценарий | Условие | Решение |
|---|---|---|
| Успех | CTR значимо вырос и положительный эффект не ниже MDE; guardrail не ухудшились | Внедряем нейросетевой ранкер |
| Провал по эффекту | CTR не вырос или значимо снизился | Оставляем CatBoost |
| Провал по guardrail | CTR вырос, но guardrail значимо и практически ухудшились | Оставляем CatBoost, фиксируем проблему |
| Неопределённость | Разница статистически незначима | Повторяем тест с большей мощностью или меняем гипотезу |

Решение принимается только по этой таблице, без исключений по отдельным метрикам или сегментам.


## 9. Риски и ограничения

### 9.1 Методологические

- **Изменение пользовательского поведения после запуска.** Первые дни могут отличаться от последующих, поэтому результат одного дня не рассматривается как достаточный.
- **Эффект сезонности.** Праздники, премьеры, маркетинговые кампании в период теста могут исказить CTR. Известные события фиксируются при интерпретации.
- **Взаимное влияние групп.** Если пользователь может попасть в обе группы (например, при разных устройствах без единого `user_id`), сравнение становится невалидным. Решается фиксацией варианта по `user_id`.

### 9.2 Технические

- **SRM.** Баг в рандомизации или потеря данных может сместить распределение групп. Проверяется в первую очередь; при обнаружении тест не анализируется до выяснения причины.
- **Задержка логирования.** События кликов и показов могут доходить с опозданием, особенно на мобильных. Анализ запускается после завершения окна дозревания данных.
- **Время загрузки нейросети.** Новая модель может быть тяжелее и влиять на UX независимо от качества рекомендаций. Это guardrail-метрика и одновременно источник риска провала теста.

### 9.3 Продуктовые

- **Локальный оптимум.** Рост CTR не гарантирует рост просмотров или удержания: пользователь может кликать, но не смотреть. Это отслеживается через secondary-метрики и retention.
- **Смещение в пользу «кликбейта».** Нейросеть может повысить клики без пропорционального роста потребления контента. Косвенно это видно через просмотры, жалобы и retention.

### 9.4 Ограничения интерпретации

- **Один раздел.** Результат относится только к ленте «Что смотреть». Переносить вывод на другие разделы или на главную страницу нельзя без отдельного теста.
- **Одна метрика решения.** Решение принимается по CTR. Если продукт позже сменит приоритет, дизайн теста придётся пересматривать.
- **Нет оценки долгосрочного эффекта.** Двух недель достаточно для краткосрочного эффекта, но недостаточно для вывода о влиянии на LTV или годовой retention.


## 10. Итог

Спроектирован A/B-тест для сравнения нейросетевого ранкера с текущим CatBoost в разделе «Что смотреть» онлайн-кинотеатра.

- **Гипотеза:** нейросетевой ранкер увеличит CTR рекомендаций минимум на +5% относительно без ухудшения guardrail-метрик.
- **Дизайн:** A/B, рандомизация по `user_id`, 50/50, длительность 2 недели.
- **Выборка:** 25 580 пользователей на группу (51 160 всего) — расчётно набирается за 1 день при трафике 200 000 польз./день.
- **Логирование:** вариант эксперимента и ключевые пользовательские события фиксируются на уровне `user_id`.
- **Retention:** после окончания теста выдерживается дополнительное 7-дневное окно дозревания.
- **Решение:** принимается по заранее зафиксированным критериям primary и guardrail-метрик.

Дизайн эксперимента готов к согласованию и реализации в продукте.
